In [1]:
# ============================================================
# MIMICEL Core Distribution EDA
# Target:
# 1. Activity Distribution
# 2. Trace Length Distribution
# 3. Case Duration Distribution
# 4. Inter-Event Time Distribution
# 5. DFG Transition Distribution
# 6. Trace Start/End Activity Distribution
# ============================================================

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 0. Config
# ------------------------------------------------------------

DATA_PATH = "./../MIMICEL_data/mimicel_train.csv"
OUT_DIR = Path("./results/train")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CASE_COL = "stay_id"
TIME_COL = "timestamps"
ACT_COL = "activity"

TOP_N = 30


# ------------------------------------------------------------
# Helper Functions
# ------------------------------------------------------------

def annotate_vertical_bars(bars, fontsize=8, rotation=90):
    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            height,
            f"{int(height):,}",
            ha="center",
            va="bottom",
            fontsize=fontsize,
            rotation=rotation
        )


def annotate_horizontal_bars(bars, fontsize=8):
    for bar in bars:
        width = bar.get_width()
        plt.text(
            width,
            bar.get_y() + bar.get_height() / 2,
            f"{int(width):,}",
            va="center",
            fontsize=fontsize
        )


def add_mean_median_lines(series):
    mean_val = series.mean()
    median_val = series.median()

    plt.axvline(
        mean_val,
        linestyle="--",
        label=f"Mean={mean_val:.2f}"
    )

    plt.axvline(
        median_val,
        linestyle="-.",
        label=f"Median={median_val:.2f}"
    )

    plt.legend()


# ------------------------------------------------------------
# 1. Load Core Columns
# ------------------------------------------------------------

print("[1] Loading data...")

df = pd.read_csv(
    DATA_PATH,
    usecols=[CASE_COL, TIME_COL, ACT_COL],
    low_memory=False
)

print("Shape:", df.shape)
print(df.head())


# ------------------------------------------------------------
# 2. Basic Preprocessing
# ------------------------------------------------------------

print("[2] Preprocessing...")

df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")

df = df.dropna(subset=[CASE_COL, TIME_COL, ACT_COL])
df = df.sort_values([CASE_COL, TIME_COL]).reset_index(drop=True)

print("After preprocessing:", df.shape)


# ============================================================
# 3. Activity Distribution
# ============================================================

print("[3] Activity Distribution...")

activity_dist = (
    df[ACT_COL]
    .value_counts()
    .reset_index()
)

activity_dist.columns = [ACT_COL, "count"]
activity_dist["ratio"] = activity_dist["count"] / activity_dist["count"].sum()

activity_dist.to_csv(
    OUT_DIR / "activity_distribution.csv",
    index=False
)

activity_top = activity_dist.head(TOP_N)

plt.figure(figsize=(12, 5))
bars = plt.bar(
    activity_top[ACT_COL],
    activity_top["count"]
)

annotate_vertical_bars(bars)

plt.xticks(rotation=45, ha="right")
plt.title("Activity Distribution")
plt.xlabel("Activity")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(
    OUT_DIR / "activity_distribution.png",
    dpi=300
)
plt.close()


# ============================================================
# 4. Trace Length Distribution
# ============================================================

print("[4] Trace Length Distribution...")

trace_len = (
    df.groupby(CASE_COL)
    .size()
    .reset_index(name="trace_length")
)

trace_len.to_csv(
    OUT_DIR / "trace_length_by_case.csv",
    index=False
)

trace_len_summary = trace_len["trace_length"].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
)

trace_len_summary.to_csv(
    OUT_DIR / "trace_length_summary.csv"
)

plt.figure(figsize=(10, 5))
plt.hist(trace_len["trace_length"], bins=100)
add_mean_median_lines(trace_len["trace_length"])
plt.title("Trace Length Distribution")
plt.xlabel("Number of Events per Case")
plt.ylabel("Number of Cases")
plt.tight_layout()
plt.savefig(
    OUT_DIR / "trace_length_distribution.png",
    dpi=300
)
plt.close()

plt.figure(figsize=(10, 5))
plt.hist(trace_len["trace_length"], bins=100)
plt.yscale("log")
add_mean_median_lines(trace_len["trace_length"])
plt.title("Trace Length Distribution - Log Scale")
plt.xlabel("Number of Events per Case")
plt.ylabel("Number of Cases - log scale")
plt.tight_layout()
plt.savefig(
    OUT_DIR / "trace_length_distribution_log.png",
    dpi=300
)
plt.close()


# ============================================================
# 5. Case Duration Distribution
# ============================================================

print("[5] Case Duration Distribution...")

case_duration = (
    df.groupby(CASE_COL)[TIME_COL]
    .agg(start_time="min", end_time="max")
    .reset_index()
)

case_duration["duration_hour"] = (
    case_duration["end_time"] - case_duration["start_time"]
).dt.total_seconds() / 3600

case_duration.to_csv(
    OUT_DIR / "case_duration_by_case.csv",
    index=False
)

case_duration_summary = case_duration["duration_hour"].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
)

case_duration_summary.to_csv(
    OUT_DIR / "case_duration_summary.csv"
)

plt.figure(figsize=(10, 5))
plt.hist(case_duration["duration_hour"], bins=100)
add_mean_median_lines(case_duration["duration_hour"])
plt.title("Case Duration Distribution")
plt.xlabel("Duration Hours")
plt.ylabel("Number of Cases")
plt.tight_layout()
plt.savefig(
    OUT_DIR / "case_duration_distribution.png",
    dpi=300
)
plt.close()

plt.figure(figsize=(10, 5))
plt.hist(case_duration["duration_hour"], bins=100)
plt.yscale("log")
add_mean_median_lines(case_duration["duration_hour"])
plt.title("Case Duration Distribution - Log Scale")
plt.xlabel("Duration Hours")
plt.ylabel("Number of Cases - log scale")
plt.tight_layout()
plt.savefig(
    OUT_DIR / "case_duration_distribution_log.png",
    dpi=300
)
plt.close()


# ============================================================
# 6. Inter-Event Time Distribution
# ============================================================

print("[6] Inter-Event Time Distribution...")

df["prev_time"] = df.groupby(CASE_COL)[TIME_COL].shift(1)

df["inter_event_minutes"] = (
    df[TIME_COL] - df["prev_time"]
).dt.total_seconds() / 60

inter_event = df.dropna(subset=["inter_event_minutes"]).copy()
inter_event = inter_event[inter_event["inter_event_minutes"] >= 0]

inter_event[[
    CASE_COL,
    TIME_COL,
    ACT_COL,
    "prev_time",
    "inter_event_minutes"
]].to_csv(
    OUT_DIR / "inter_event_time.csv",
    index=False
)

inter_event_summary = inter_event["inter_event_minutes"].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
)

inter_event_summary.to_csv(
    OUT_DIR / "inter_event_time_summary.csv"
)

plt.figure(figsize=(10, 5))
plt.hist(inter_event["inter_event_minutes"], bins=100)
add_mean_median_lines(inter_event["inter_event_minutes"])
plt.title("Inter-Event Time Distribution")
plt.xlabel("Minutes Between Events")
plt.ylabel("Number of Events")
plt.tight_layout()
plt.savefig(
    OUT_DIR / "inter_event_time_distribution.png",
    dpi=300
)
plt.close()

plt.figure(figsize=(10, 5))
plt.hist(inter_event["inter_event_minutes"], bins=100)
plt.yscale("log")
add_mean_median_lines(inter_event["inter_event_minutes"])
plt.title("Inter-Event Time Distribution - Log Scale")
plt.xlabel("Minutes Between Events")
plt.ylabel("Number of Events - log scale")
plt.tight_layout()
plt.savefig(
    OUT_DIR / "inter_event_time_distribution_log.png",
    dpi=300
)
plt.close()


# ============================================================
# 7. DFG Transition Distribution
# ============================================================

print("[7] DFG Transition Distribution...")

df["next_activity"] = df.groupby(CASE_COL)[ACT_COL].shift(-1)

dfg = (
    df.dropna(subset=["next_activity"])
    .groupby([ACT_COL, "next_activity"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

dfg["ratio"] = dfg["count"] / dfg["count"].sum()

dfg.to_csv(
    OUT_DIR / "dfg_transition_distribution.csv",
    index=False
)

dfg_top = dfg.head(TOP_N).copy()
dfg_top["transition"] = (
    dfg_top[ACT_COL] + " -> " + dfg_top["next_activity"]
)

plt.figure(figsize=(12, 6))
bars = plt.barh(
    dfg_top["transition"][::-1],
    dfg_top["count"][::-1]
)

annotate_horizontal_bars(bars)

plt.title("Top DFG Transitions")
plt.xlabel("Count")
plt.ylabel("Transition")
plt.tight_layout()
plt.savefig(
    OUT_DIR / "dfg_transition_distribution.png",
    dpi=300
)
plt.close()


# ============================================================
# 8. Trace Start/End Activity Distribution
# ============================================================

print("[8] Trace Start/End Activity Distribution...")

first_last = (
    df.groupby(CASE_COL)
    .agg(
        start_activity=(ACT_COL, "first"),
        end_activity=(ACT_COL, "last")
    )
    .reset_index()
)

start_dist = (
    first_last["start_activity"]
    .value_counts()
    .reset_index()
)

start_dist.columns = ["start_activity", "count"]
start_dist["ratio"] = start_dist["count"] / start_dist["count"].sum()

end_dist = (
    first_last["end_activity"]
    .value_counts()
    .reset_index()
)

end_dist.columns = ["end_activity", "count"]
end_dist["ratio"] = end_dist["count"] / end_dist["count"].sum()

first_last.to_csv(
    OUT_DIR / "trace_start_end_by_case.csv",
    index=False
)

start_dist.to_csv(
    OUT_DIR / "trace_start_activity_distribution.csv",
    index=False
)

end_dist.to_csv(
    OUT_DIR / "trace_end_activity_distribution.csv",
    index=False
)

start_top = start_dist.head(TOP_N)

plt.figure(figsize=(10, 5))
bars = plt.bar(
    start_top["start_activity"],
    start_top["count"]
)

annotate_vertical_bars(bars)

plt.xticks(rotation=45, ha="right")
plt.title("Trace Start Activity Distribution")
plt.xlabel("Start Activity")
plt.ylabel("Number of Cases")
plt.tight_layout()
plt.savefig(
    OUT_DIR / "trace_start_activity_distribution.png",
    dpi=300
)
plt.close()

end_top = end_dist.head(TOP_N)

plt.figure(figsize=(10, 5))
bars = plt.bar(
    end_top["end_activity"],
    end_top["count"]
)

annotate_vertical_bars(bars)

plt.xticks(rotation=45, ha="right")
plt.title("Trace End Activity Distribution")
plt.xlabel("End Activity")
plt.ylabel("Number of Cases")
plt.tight_layout()
plt.savefig(
    OUT_DIR / "trace_end_activity_distribution.png",
    dpi=300
)
plt.close()


# ============================================================
# 9. Summary Report
# ============================================================

print("[9] Writing summary report...")

with open(
    OUT_DIR / "eda_summary.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write("MIMICEL Core Distribution EDA Summary\n")
    f.write("=" * 60 + "\n\n")

    f.write("[Dataset]\n")
    f.write(f"Rows: {len(df):,}\n")
    f.write(f"Cases: {df[CASE_COL].nunique():,}\n")
    f.write(f"Activities: {df[ACT_COL].nunique():,}\n\n")

    f.write("[Activity Distribution Top]\n")
    f.write(activity_dist.head(20).to_string(index=False))
    f.write("\n\n")

    f.write("[Trace Length Summary]\n")
    f.write(trace_len_summary.to_string())
    f.write("\n\n")

    f.write("[Case Duration Summary]\n")
    f.write(case_duration_summary.to_string())
    f.write("\n\n")

    f.write("[Inter-Event Time Summary]\n")
    f.write(inter_event_summary.to_string())
    f.write("\n\n")

    f.write("[DFG Top Transitions]\n")
    f.write(dfg.head(20).to_string(index=False))
    f.write("\n\n")

    f.write("[Start Activity Distribution]\n")
    f.write(start_dist.head(20).to_string(index=False))
    f.write("\n\n")

    f.write("[End Activity Distribution]\n")
    f.write(end_dist.head(20).to_string(index=False))
    f.write("\n\n")

print("Done.")
print(f"Outputs saved to: {OUT_DIR.resolve()}")

[1] Loading data...
Shape: (5293114, 3)
    stay_id           timestamps                 activity
0  30000012  2126-02-14 20:22:00         Vital sign check
1  30000012  2126-02-14 20:22:00             Enter the ED
2  30000012  2126-02-14 20:22:01         Triage in the ED
3  30000012  2126-02-14 22:21:00  Medicine reconciliation
4  30000012  2126-02-14 22:21:00  Medicine reconciliation
[2] Preprocessing...
After preprocessing: (5293114, 3)
[3] Activity Distribution...
[4] Trace Length Distribution...
[5] Case Duration Distribution...
[6] Inter-Event Time Distribution...
[7] DFG Transition Distribution...
[8] Trace Start/End Activity Distribution...
[9] Writing summary report...
Done.
Outputs saved to: D:\University\4-1\6Process_Mining\PM_Assessment_Synthetic_Event_Logs\02_Data_EDA\results\train
